In [1]:
import requests
import json
import pandas as pd
from getpass import getpass

In [2]:
from dbrepo.RestClient import RestClient
from getpass import getpass

username = "Bernh"
password = "vA3^#&8Dy]#8:N2"
#password = getpass()
auth = (username, password)

headers = {
    "Content-Type": "application/json"
}

BASE_URL = "https://test.dbrepo.tuwien.ac.at/api/v1"

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username=username,
    password=password
)

containers = client.get_containers()

for c in containers:
    print(c)

id='6cfb3b8e-1792-4e46-871a-f3d103527203' name='mariadb-galera:11.3.2' image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False) internal_name='mariadb_11_3_2' running=None hash=None


In [6]:
# /api/v1/database/38707917-e942-45c3-a3dd-d2bfc1c106af/tables

database_id = "38707917-e942-45c3-a3dd-d2bfc1c106af"

tables = client.get_tables(database_id)

for db in tables:
    print(db)

id='8882a560-da44-4bb9-b14d-28ed91512c4b' database_id='38707917-e942-45c3-a3dd-d2bfc1c106af' name='Population_statistics' description=None internal_name='population_statistics' is_versioned=True is_public=True is_schema_public=True owned_by='bernh'
id='b4a40404-3f22-4d6f-a534-8bae624aa576' database_id='38707917-e942-45c3-a3dd-d2bfc1c106af' name='Time_dimension' description='Temporal dimension table representing yearly reference dates' internal_name='time_dimension' is_versioned=True is_public=True is_schema_public=True owned_by='bernh'
id='19a15041-62dc-4676-992a-ea5acdb71ce3' database_id='38707917-e942-45c3-a3dd-d2bfc1c106af' name='Nationality_group' description='Lookup table containing grouped nationality categories used in demographic statistics' internal_name='nationality_group' is_versioned=True is_public=True is_schema_public=True owned_by='bernh'
id='a2333bf4-4ce2-42e6-a11d-32f9213b6488' database_id='38707917-e942-45c3-a3dd-d2bfc1c106af' name='Age_group' description='Lookup tabl

In [11]:
population_statistics_table_id = "8882a560-da44-4bb9-b14d-28ed91512c4b"

columns = client.get_table(database_id, population_statistics_table_id)

print(columns)

id='8882a560-da44-4bb9-b14d-28ed91512c4b' database_id='38707917-e942-45c3-a3dd-d2bfc1c106af' name='Population_statistics' owner=UserBrief(username='bernh', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='bf624224-96c2-41a5-9cb8-cb05a3f25ad5', name='population_count', database_id='38707917-e942-45c3-a3dd-d2bfc1c106af', table_id='8882a560-da44-4bb9-b14d-28ed91512c4b', ord=0, internal_name='population_count', is_null_allowed=False, type=<ColumnType.INT: 'int'>, alias=None, description='Observed population count', size=None, d=None, mean=None, median=None, concept=None, unit=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='a8ab94db-35ba-43bd-88bf-583f51633dec', name='nationality_code', database_id='38707917-e942-45c3-a3dd-d2bfc1c106af', table_id='8882a560-da44-4bb9-b14d-28ed91512c4b', ord=1, internal_name='nationali

In [15]:
payload_population_pyramid_base = {
    "name": "population_pyramid_base",
    "query": {
        "datasource_ids": [
            "8882a560-da44-4bb9-b14d-28ed91512c4b",  # Population_statistics
            "a2333bf4-4ce2-42e6-a11d-32f9213b6488"   # Age_group
        ],

        "columns": [
            {
                "id": "1fb79e29-512a-48d8-a101-7e489d345ac0",
                "alias": "year"
            },
            {
                "id": "ccea4d66-0c11-4b57-824a-b298791aff80",
                "alias": "age_group_id"
            },
            {
                "id": "6529b7af-6734-4cb2-81be-db7970596461",
                "alias": "age_group_lookup_id"
            },
            {
                "id": "cf085a81-9f5d-4a6a-a0f9-2695111fd913",
                "alias": "sex_id"
            },
            {
                "id": "bf624224-96c2-41a5-9cb8-cb05a3f25ad5",
                "alias": "population_count"
            }
        ],

        "joins": [
            {
                "type": "inner",
                "datasource_id": "a2333bf4-4ce2-42e6-a11d-32f9213b6488",
                "conditionals": [
                    {
                        "column_id": "ccea4d66-0c11-4b57-824a-b298791aff80",
                        "foreign_column_id": "6529b7af-6734-4cb2-81be-db7970596461"
                    }
                ]
            }
        ],

        "filters": [],

        "orders": [
            {
                "direction": "asc",
                "column_id": "1fb79e29-512a-48d8-a101-7e489d345ac0"
            }
        ]
    },

    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/view",
    auth=auth,
    headers=headers,
    json=payload_population_pyramid_base
)

print(response.status_code)
print(response.text)

201
{"id":"40d0e45f-3e41-4c0d-8c2e-41d8967be4c2","name":"population_pyramid_base","query":"select `vienna_demographic_forecasting_2e4d`.`population_statistics`.`age_group_id` as `age_group_id`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`year` as `year`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`population_count` as `population_count`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`sex_id` as `sex_id`, `vienna_demographic_forecasting_2e4d`.`age_group`.`age_group_id` as `age_group_lookup_id` from `age_group`, `population_statistics` join `age_group` on `vienna_demographic_forecasting_2e4d`.`population_statistics`.`age_group_id` = `vienna_demographic_forecasting_2e4d`.`age_group`.`age_group_id` order by `vienna_demographic_forecasting_2e4d`.`population_statistics`.`year` asc","database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"population_pyramid_base","is_public":true,"is_schema_public":true,"initial_view":false,"q

In [17]:
payload_population_by_nationality = {
    "name": "population_by_nationality",

    "query": {
        "datasource_ids": [
            "8882a560-da44-4bb9-b14d-28ed91512c4b"
        ],

        "columns": [
            {
                "id": "1fb79e29-512a-48d8-a101-7e489d345ac0",
                "alias": "year"
            },
            {
                "id": "a8ab94db-35ba-43bd-88bf-583f51633dec",
                "alias": "nationality_code"
            },
            {
                "id": "bf624224-96c2-41a5-9cb8-cb05a3f25ad5",
                "alias": "population_count"
            }
        ],

        "joins": [],
        "filters": [],

        "orders": [
            {
                "direction": "asc",
                "column_id": "1fb79e29-512a-48d8-a101-7e489d345ac0"
            }
        ]
    },

    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/view",
    auth=auth,
    headers=headers,
    json=payload_population_by_nationality
)

#columns = client.create_view(database_id, payload)

print(response.status_code)
print(response.text)

201
{"id":"9517ff56-7559-4999-88db-0236bcffcdd4","name":"population_by_nationality","query":"select `vienna_demographic_forecasting_2e4d`.`population_statistics`.`year` as `year`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`nationality_code` as `nationality_code`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`population_count` as `population_count` from `population_statistics` order by `vienna_demographic_forecasting_2e4d`.`population_statistics`.`year` asc","database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"population_by_nationality","is_public":true,"is_schema_public":true,"initial_view":false,"query_hash":"fa81ecab2b00fe83e2cd5c8d007e05d6c32b82535b39bbe395944be01e075fb7","owned_by":"bernh"}


In [18]:
payload_population_by_district = {
    "name": "population_by_district",

    "query": {
        "datasource_ids": [
            "8882a560-da44-4bb9-b14d-28ed91512c4b"
        ],

        "columns": [
            {
                "id": "1fb79e29-512a-48d8-a101-7e489d345ac0",
                "alias": "year"
            },
            {
                "id": "17c852b3-5803-4d59-8abf-27bb8840b44f",
                "alias": "district_code"
            },
            {
                "id": "cf085a81-9f5d-4a6a-a0f9-2695111fd913",
                "alias": "sex_id"
            },
            {
                "id": "ccea4d66-0c11-4b57-824a-b298791aff80",
                "alias": "age_group_id"
            },
            {
                "id": "a8ab94db-35ba-43bd-88bf-583f51633dec",
                "alias": "nationality_code"
            },
            {
                "id": "bf624224-96c2-41a5-9cb8-cb05a3f25ad5",
                "alias": "population_count"
            }
        ],

        "joins": [],
        "filters": [],

        "orders": [
            {
                "direction": "asc",
                "column_id": "1fb79e29-512a-48d8-a101-7e489d345ac0"
            }
        ]
    },

    "is_public": True,
    "is_schema_public": True
}

response = requests.post(
    f"{BASE_URL}/database/{database_id}/view",
    auth=auth,
    headers=headers,
    json=payload_population_by_district
)

print(response.status_code)
print(response.text)

201
{"id":"1448ed5a-3777-482c-8857-ea83a590f416","name":"population_by_district","query":"select `vienna_demographic_forecasting_2e4d`.`population_statistics`.`age_group_id` as `age_group_id`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`district_code` as `district_code`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`year` as `year`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`nationality_code` as `nationality_code`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`population_count` as `population_count`, `vienna_demographic_forecasting_2e4d`.`population_statistics`.`sex_id` as `sex_id` from `population_statistics` order by `vienna_demographic_forecasting_2e4d`.`population_statistics`.`year` asc","database_id":"38707917-e942-45c3-a3dd-d2bfc1c106af","internal_name":"population_by_district","is_public":true,"is_schema_public":true,"initial_view":false,"query_hash":"61540f383f01250ecc6665360d2466deefa28e11c0f796b74ad8c447